In [1]:
# ============================================================
# 金融新闻情绪分析器
# ============================================================
import nltk
from nltk.corpus import wordnet as wn
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import word_tokenize, sent_tokenize
from collections import Counter


# ── 1. 示例财经新闻 ──────────────────────────────────────────
news_samples = {
    "利好消息": """
        Apple reported record quarterly earnings, beating analyst expectations.
        Revenue surged 15% driven by strong iPhone sales and services growth.
        The stock rallied sharply after the bullish guidance for next quarter.
        Investors are optimistic about the company's expansion into AI products.
    """,
    "利空消息": """
        The Federal Reserve raised interest rates again, triggering a market selloff.
        Bank stocks plunged as fears of recession intensified.
        Unemployment data disappointed, signaling weakness in the labor market.
        Several hedge funds reported massive losses amid the volatile trading session.
    """,
    "中性消息": """
        The company announced its quarterly results in line with market expectations.
        Revenue remained stable compared to the previous period.
        Management provided guidance consistent with analyst forecasts.
        Trading volume was average with no significant price movement observed.
    """
}

# ── 2. WordNet 金融关键词语义扩展 ────────────────────────────
def get_financial_synonyms(word):
    """利用 WordNet 获取金融词汇的同义词集"""
    synsets = wn.synsets(word)
    synonyms = set()
    for syn in synsets[:2]:  # 只取前2个最相关的同义词集
        for lemma in syn.lemmas():
            synonyms.add(lemma.name().replace('_', ' '))
    return synonyms

def get_hypernym_chain(word):
    """获取词的上位词链（概念层级）"""
    synsets = wn.synsets(word, pos=wn.NOUN)
    if not synsets:
        return []
    chain = []
    current = synsets[0]
    for _ in range(4):  # 往上追溯4层
        hypernyms = current.hypernyms()
        if not hypernyms:
            break
        current = hypernyms[0]
        chain.append(current.lemma_names()[0])
    return chain

# 金融核心词汇 + WordNet 扩展
core_fin_words = ['profit', 'loss', 'revenue', 'stock', 'market', 'bank', 'investment']

print("=" * 55)
print("  📚 WordNet 金融词汇语义网络")
print("=" * 55)
for word in core_fin_words:
    syns = get_financial_synonyms(word)
    chain = get_hypernym_chain(word)
    syns_display = ', '.join(list(syns)[:4]) if syns else 'N/A'
    chain_display = ' → '.join(chain) if chain else 'N/A'
    print(f"\n🔑 [{word}]")
    print(f"   同义词: {syns_display}")
    print(f"   上位词: {chain_display}")

# ── 3. 情绪分析 ──────────────────────────────────────────────
sia = SentimentIntensityAnalyzer()

# 自定义金融情绪词权重（增强 VADER 对财经词的敏感度）
finance_lexicon = {
    'surge':  2.5, 'rally':   2.0, 'bullish':  2.8, 'record':   1.5,
    'beat':   1.8, 'growth':  1.5, 'optimistic': 2.0,
    'plunge': -2.5, 'selloff': -2.8, 'recession': -2.5, 'loss': -1.8,
    'volatile': -1.2, 'disappoint': -2.0, 'fear': -1.8, 'weak': -1.5,
}
sia.lexicon.update(finance_lexicon)

def analyze_news(title, text):
    """对一段财经文本做完整分析"""
    sentences = sent_tokenize(text.strip())
    words = word_tokenize(text.lower())

    # 过滤停用词，统计高频词
    stopwords = set(nltk.corpus.stopwords.words('english'))
    meaningful_words = [w for w in words if w.isalpha() and w not in stopwords]
    top_words = Counter(meaningful_words).most_common(5)

    # 逐句情绪打分
    sent_scores = []
    for sent in sentences:
        if sent.strip():
            score = sia.polarity_scores(sent)
            sent_scores.append((sent.strip(), score['compound']))

    # 整体情绪
    overall = sia.polarity_scores(text)
    compound = overall['compound']
    if compound >= 0.05:
        label, emoji = "正面 (Bullish)", "📈"
    elif compound <= -0.05:
        label, emoji = "负面 (Bearish)", "📉"
    else:
        label, emoji = "中性 (Neutral)", "➡️"

    # 输出报告
    print(f"\n{'=' * 55}")
    print(f"  {emoji}  {title}")
    print(f"{'=' * 55}")
    print(f"  综合情绪得分: {compound:+.4f}  →  {label}")
    print(f"  正面={overall['pos']:.2f}  负面={overall['neg']:.2f}  中性={overall['neu']:.2f}")
    print(f"\n  📊 高频关键词: {', '.join([w for w,_ in top_words])}")
    print(f"\n  📝 逐句得分:")
    for sent, score in sent_scores:
        bar = "▓" * int(abs(score) * 10)
        sign = "+" if score > 0 else ""
        print(f"    [{sign}{score:.2f}] {bar}  {sent[:60]}...")

print("\n" + "=" * 55)
print("  📰 金融新闻情绪分析报告")
print("=" * 55)
for title, text in news_samples.items():
    analyze_news(title, text)

print("\n✅ 分析完成")

  📚 WordNet 金融词汇语义网络

🔑 [profit]
   同义词: lucre, net, profit, gain
   上位词: income → financial_gain → gain → sum

🔑 [loss]
   同义词: loss
   上位词: transferred_property → possession → relation → abstraction

🔑 [revenue]
   同义词: receipts, revenue, gross, tax revenue
   上位词: sum → assets → possession → relation

🔑 [stock]
   同义词: stock, inventory
   上位词: capital → assets → possession → relation

🔑 [market]
   同义词: market, marketplace, market place
   上位词: activity → act → event → psychological_feature

🔑 [bank]
   同义词: banking concern, depository financial institution, banking company, bank
   上位词: slope → geological_formation → object → physical_entity

🔑 [investment]
   同义词: investment, investment funds, investing
   上位词: finance → commercial_enterprise → commerce → transaction

  📰 金融新闻情绪分析报告

  📈  利好消息
  综合情绪得分: +0.9022  →  正面 (Bullish)
  正面=0.29  负面=0.06  中性=0.65

  📊 高频关键词: apple, reported, record, quarterly, earnings

  📝 逐句得分:
    [-0.13] ▓  Apple reported record quarterly earnings, be